# 📡 Advanced Forecasting
**fpppy Chapters 10–12 · Data Pattern Recognition for the Rest of Us**

> Prophet handles complex business seasonality, promotions, and holidays automatically. ML-based forecasting with lag features scales to thousands of SKUs. This notebook shows the modern retail forecasting toolkit.

### Required reading
📘 [fpppy Ch 10–12](https://otexts.com/fpppy/)

### Dataset
**Retail Store Sales** — weekly sales for a multi-department retailer with promotional events, holidays, and seasonal patterns. Modelled on M5 competition and Walmart sales data characteristics.

### Time: ~70 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install prophet -q
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
print("\u2713 Setup complete — Prophet installed")

In [ ]:
# Retail Store Weekly Sales — realistic M5-style dataset
# Features: trend, weekly/annual seasonality, holiday spikes, promotions
np.random.seed(42)

dates = pd.date_range('2018-01-01', '2023-12-31', freq='W-MON')
n = len(dates)

# Trend: slow growth
trend = 50000 + np.linspace(0, 20000, n)

# Annual seasonality: back-to-school (Aug), holiday (Nov-Dec), post-holiday drop (Jan)
doy = np.array([d.timetuple().tm_yday for d in dates])
annual = (8000 * np.sin(2*np.pi*(doy-80)/365)
         + 4000 * np.sin(4*np.pi*(doy-40)/365))

# Weekly seasonality already in weekly data (aggregated)

# Holiday spikes
holiday_boost = np.zeros(n)
for i, d in enumerate(dates):
    if d.month == 11 and d.day >= 20:   # Thanksgiving week
        holiday_boost[i] = 25000
    elif d.month == 12 and d.day <= 27: # Christmas
        holiday_boost[i] = 35000
    elif d.month == 7 and 1<=d.day<=7:  # July 4th
        holiday_boost[i] = 8000
    elif d.month == 1 and d.day <= 7:   # Post-holiday drop
        holiday_boost[i] = -15000

# Promotions (random but correlated with slow periods)
promotion = np.random.choice([0, 5000, 12000, 20000], n, p=[0.6, 0.2, 0.12, 0.08])

# Noise
noise = np.random.normal(0, 3500, n)

# Final sales
sales = (trend + annual + holiday_boost + promotion + noise).clip(10000)

retail = pd.DataFrame({'ds': dates, 'y': sales.round(-2),
                        'promotion': (promotion > 0).astype(int)})

print(f"Retail Sales dataset: {len(retail)} weekly observations")
print(f"Date range: {retail['ds'].min().date()} to {retail['ds'].max().date()}")
print(f"Weekly sales range: ${retail['y'].min():,.0f} — ${retail['y'].max():,.0f}")
print(f"Median weekly sales: ${retail['y'].median():,.0f}")
print(f"Promotion weeks: {retail['promotion'].sum()} ({retail['promotion'].mean():.1%})")

In [ ]:
# Exploratory: visualize the components we built in
fig, axes = plt.subplots(3, 1, figsize=(13, 9))

axes[0].plot(retail['ds'], retail['y']/1000, color='#1e5fa8', lw=1.2, alpha=0.8)
axes[0].set_title('Weekly Retail Sales — Multi-Year View')
axes[0].set_ylabel('Sales ($000s)')

# Zoom into one year
one_year = retail[retail['ds'].dt.year == 2022]
axes[1].plot(one_year['ds'], one_year['y']/1000, color='#e85d20', lw=1.5)
promo_weeks = one_year[one_year['promotion']==1]
axes[1].scatter(promo_weeks['ds'], promo_weeks['y']/1000,
                color='#1a7a45', s=60, zorder=3, label='Promotion week')
axes[1].set_title('2022 Weekly Sales — Seasonal Pattern + Promotions')
axes[1].set_ylabel('Sales ($000s)')
axes[1].legend()

# Year-over-year comparison
for year in [2020, 2021, 2022, 2023]:
    yr_data = retail[retail['ds'].dt.year == year]
    if len(yr_data) >= 50:
        axes[2].plot(range(len(yr_data)), yr_data['y']/1000,
                    lw=1.5, alpha=0.8, label=str(year))
axes[2].set_title('Year-over-Year Comparison')
axes[2].set_xlabel('Week of Year')
axes[2].set_ylabel('Sales ($000s)')
axes[2].legend()

plt.tight_layout(); plt.show()

## 🔮 Part 1 — Prophet: Business Forecasting with Seasonality & Events

Prophet decomposes sales as:
```
y(t) = trend(t) + weekly_seasonality(t) + annual_seasonality(t) + holidays(t) + ε
```

The holiday/promotion component is critical for retail — holiday weeks are 40-60% above baseline, and models that ignore them produce terrible forecasts.

In [ ]:
# Temporal train/test split — never shuffle time series
split_date = '2023-01-01'
train = retail[retail['ds'] < split_date][['ds','y']]
test  = retail[retail['ds'] >= split_date][['ds','y']]
h = len(test)

print(f"Training: {len(train)} weeks ({train['ds'].min().date()} to {train['ds'].max().date()})")
print(f"Test:     {h} weeks ({test['ds'].min().date()} to {test['ds'].max().date()})")

# Define retail holidays
from prophet.make_holidays import make_holidays_df
# Manual holiday list for retail
holidays = pd.DataFrame({
    'holiday': ['thanksgiving','christmas','july4','postthanksgiving','post_holiday'],
    'ds': pd.to_datetime([
        '2018-11-22','2019-11-28','2020-11-26','2021-11-25','2022-11-24',
        '2018-12-24','2019-12-23','2020-12-21','2021-12-20','2022-12-19',
        '2018-07-02','2019-07-01','2020-06-29','2021-06-28','2022-07-04',
        '2018-11-26','2019-12-02','2020-11-30','2021-11-29','2022-11-28',
        '2019-01-07','2020-01-06','2021-01-04','2022-01-03','2023-01-02',
    ]),
    'lower_window': [0,0,0,0,-2],
    'upper_window': [1,1,1,2,0]
})

# Fit Prophet
m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
            holidays=holidays,
            changepoint_prior_scale=0.08,
            seasonality_prior_scale=15,
            holidays_prior_scale=20)
m.fit(train)

future = m.make_future_dataframe(periods=h, freq='W')
forecast = m.predict(future)
test_forecast = forecast[forecast['ds'].isin(test['ds'])]

mae_prophet = mean_absolute_error(test['y'], test_forecast['yhat'])
mape_prophet= (abs(test['y'].values - test_forecast['yhat'].values) / test['y'].values).mean()

print(f"\nProphet test performance:")
print(f"  MAE:  ${mae_prophet:,.0f}")
print(f"  MAPE: {mape_prophet:.1%}")

In [ ]:
# Prophet forecast plot
fig = m.plot(forecast, figsize=(13, 5))
fig.axes[0].set_title('Prophet Forecast — Retail Weekly Sales')
fig.axes[0].axvline(pd.Timestamp(split_date), color='#e85d20', lw=2,
                    ls='--', label='Train/Test split')
fig.axes[0].legend()
plt.tight_layout(); plt.show()

# Components
fig2 = m.plot_components(forecast, figsize=(12, 9))
fig2.suptitle('Prophet Components: Trend + Seasonality + Holidays', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ML approach: GBM with lag features
def make_lag_features(df, target='y', lags=[1,2,4,8,13,26,52]):
    df = df.copy().set_index('ds')
    for lag in lags:
        df[f'lag_{lag}w'] = df[target].shift(lag)
    df['rolling_4w_mean'] = df[target].shift(1).rolling(4).mean()
    df['rolling_13w_mean']= df[target].shift(1).rolling(13).mean()
    df['week_of_year']    = df.index.isocalendar().week.astype(int)
    df['month']           = df.index.month
    return df.dropna()

retail_feat = make_lag_features(retail.drop('promotion',axis=1))
feature_cols= [c for c in retail_feat.columns if c != 'y']

split_idx = retail_feat.index < pd.Timestamp(split_date)
X_tr_ml = retail_feat.loc[split_idx, feature_cols]
y_tr_ml = retail_feat.loc[split_idx, 'y']
X_te_ml = retail_feat.loc[~split_idx, feature_cols]
y_te_ml = retail_feat.loc[~split_idx, 'y']

gbm_ts = GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                    learning_rate=0.05, random_state=42)
gbm_ts.fit(X_tr_ml, y_tr_ml)
y_pred_gbm = gbm_ts.predict(X_te_ml)

mae_gbm  = mean_absolute_error(y_te_ml, y_pred_gbm)
mape_gbm = (abs(y_te_ml.values - y_pred_gbm) / y_te_ml.values).mean()

# Naive baseline (same week last year)
naive_pred = retail_feat.loc[~split_idx, 'lag_52w'].values
mae_naive  = mean_absolute_error(y_te_ml, naive_pred)
mape_naive = (abs(y_te_ml.values - naive_pred) / y_te_ml.values).mean()

print("=== Model Comparison — Retail Sales Forecast ===\n")
print(f"{'Model':<25} {'MAE':>10} {'MAPE':>8}")
print("-"*45)
for name, mae, mape in [
    ('Naive (same wk last yr)', mae_naive,   mape_naive),
    ('Prophet',                 mae_prophet, mape_prophet),
    ('GBM + lag features',      mae_gbm,     mape_gbm)]:
    print(f"  {name:<23} ${mae:>9,.0f} {mape:>7.1%}")

# Feature importance for GBM
imp = pd.Series(gbm_ts.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(range(len(imp)), imp.values, color='#1e5fa8', edgecolor='white')
ax.set_xticks(range(len(imp)))
ax.set_xticklabels(imp.index, rotation=45, ha='right', fontsize=9)
ax.set_title('GBM Feature Importance — Retail Sales\n'
             'Lag features + seasonality drive predictions')
plt.tight_layout(); plt.show()

In [ ]:
# @title 📝 Quiz — Advanced Forecasting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key difference between Prophet and ARIMA?
# @markdown **Q2:** Why can you NOT randomly shuffle train/test split for time series?
# @markdown **Q3:** How do you convert a time series into a supervised ML problem?
# @markdown **Q4:** What does changepoint_prior_scale control in Prophet?
# @markdown **Q5:** For retail sales, when would you choose GBM+lags over Prophet?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Advanced Forecasting"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*